In [ ]:
"""
Word Sense Disambiguation using Lesk's Algorithm
Author: NLP Lab Submission | Date: February 2026
"""

import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download required NLTK resources
def download_resources():
    """Downloads wordnet, punkt, omw-1.4, stopwords for WSD."""
    print("Downloading NLTK resources...")
    for r in ['wordnet', 'punkt', 'omw-1.4', 'stopwords', 'punkt_tab']:
        nltk.download(r, quiet=True)
    print("Resources ready!\n")

# Preprocess text: lowercase, tokenize, remove stopwords and punctuation
def preprocess(text):
    """Returns a set of meaningful words from text."""
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    return {t for t in tokens if t.isalpha() and t not in stop_words}

# Get signature words from a sense (definition, examples, lemmas, hypernyms)
def get_signature(sense):
    """Extracts all related words from a WordNet sense."""
    sig = preprocess(sense.definition())
    
    # Add example sentence words
    for ex in sense.examples():
        sig.update(preprocess(ex))
    
    # Add lemma names (synonyms)
    for lemma in sense.lemmas():
        sig.update(preprocess(lemma.name().replace('_', ' ')))
    
    # Add hypernym definitions and lemmas
    for hyp in sense.hypernyms():
        sig.update(preprocess(hyp.definition()))
        for lemma in hyp.lemmas():
            sig.update(preprocess(lemma.name().replace('_', ' ')))
    
    return sig

# Core Lesk Algorithm
def lesk_algorithm(sentence, word):
    """
    Disambiguates word sense using Lesk Algorithm.
    Returns (best_sense_info, all_scores) or (None, None) if word not found.
    """
    senses = wn.synsets(word)
    if not senses:
        return None, None
    
    # Get context words (excluding the ambiguous word)
    context = preprocess(sentence)
    context.discard(word.lower())
    
    # Calculate overlap for each sense
    scores = []
    for sense in senses:
        sig = get_signature(sense)
        common = context & sig  # Intersection
        scores.append({
            'sense': sense,
            'score': len(common),
            'common_words': common,
            'definition': sense.definition()
        })
    
    # Select sense with maximum overlap
    best = max(scores, key=lambda x: x['score'])
    return best, scores

# Display results
def display_results(word, best, all_scores):
    """Displays disambiguation results."""
    pos_names = {'n': 'Noun', 'v': 'Verb', 'a': 'Adjective', 's': 'Adj Satellite', 'r': 'Adverb'}
    
    print(f"\n{'='*60}")
    print(f"  RESULTS FOR: '{word.upper()}'")
    print(f"{'='*60}")
    
    print(f"\n  ALL SENSES:")
    for i, s in enumerate(all_scores, 1):
        print(f"\n  {i}. {s['sense'].name()} [Score: {s['score']}]")
        print(f"     Definition: {s['definition']}")
        if s['common_words']:
            print(f"     Matches: {', '.join(s['common_words'])}")
    
    print(f"\n{'-'*60}")
    print(f"  SELECTED SENSE: {best['sense'].name()}")
    print(f"  POS: {pos_names.get(best['sense'].pos(), 'Unknown')}")
    print(f"  Definition: {best['definition']}")
    print(f"  Score: {best['score']}")
    if best['common_words']:
        print(f"  Matching Words: {', '.join(best['common_words'])}")
    print(f"{'='*60}\n")

# Main interactive program
def main():
    download_resources()
    
    print("="*60)
    print("  WORD SENSE DISAMBIGUATION - LESK'S ALGORITHM")
    print("="*60)
    print("Enter 'quit' to exit.\n")
    
    while True:
        try:
            sentence = input("Enter sentence: ").strip()
            if sentence.lower() in ['quit', 'exit', 'q']:
                break
            if not sentence:
                continue
            
            word = input("Enter ambiguous word: ").strip()
            if word.lower() in ['quit', 'exit', 'q']:
                break
            if not word:
                continue
            
            best, scores = lesk_algorithm(sentence, word)
            
            if best is None:
                print(f"Error: '{word}' not found in WordNet.\n")
                continue
            
            display_results(word, best, scores)
            
        except KeyboardInterrupt:
            break
    
    print("Goodbye!")

# Algorithm Explanation
EXPLANATION = """
HOW LESK'S ALGORITHM WORKS:
===========================
1. Get all possible senses of the ambiguous word from WordNet
2. Preprocess the context sentence (tokenize, remove stopwords)
3. For each sense, create a "signature" from:
   - Definition (gloss)
   - Example sentences
   - Synonyms (lemma names)
   - Hypernym definitions
4. Calculate overlap (common words) between context and each signature
5. Select the sense with maximum overlap

Example:
  Sentence: "I went to the bank to deposit money"
  Word: "bank"
  
  Financial sense signature: {deposit, money, account, institution...}
  River sense signature: {water, slope, shore, edge...}
  
  Context: {went, deposit, money}
  Overlap with financial: 2 (deposit, money)
  Overlap with river: 0
  
  Result: Financial sense selected
"""

if __name__ == "__main__":
    print(EXPLANATION)
    main()
